In [1]:
import sys
sys.path.insert(0, 'E:\Code\Music')

In [15]:
from pathlib import Path
from PDMX.reading.music import MusicRender
import random
import pandas as pd
from collections import Counter

In [3]:
PDMX_dir = Path('E:\Code\music\dataset')

# Data Processing

In [4]:
def extract_note_sequences(json_file: str) -> list[list]:
    note_sequences = []
    music = MusicRender()
    music.load(json_file)
      
    for track in music.tracks:
        notes = [note.pitch for note in track.notes]
        note_sequences.append(notes)
    
    return note_sequences
    

In [5]:
json_root = PDMX_dir / 'data'
json_files = [str(file) for file in json_root.rglob('*.json')]

In [6]:
all_sequences = []

for path in (random.sample(json_files, k=1000)):
    notes = extract_note_sequences(path)
    all_sequences.extend(seq for seq in notes)


In [7]:
all_notes = [note for seq in all_sequences for note in seq]
note_counts = Counter(all_notes)

vocab = sorted(note_counts.keys())
idx_to_note = {idx : note for idx, note in enumerate(vocab)}
note_to_idx = {note : idx for idx, note in idx_to_note.items()}

assert len(vocab) == len(idx_to_note) == len(note_to_idx)

VOCAB_SIZE = len(vocab)

In [8]:
pd.DataFrame.from_dict(note_counts, orient='index', columns=['count'])

,count
69,26180
78,8857
76,18607
74,23435
73,8264
...,...
25,32
24,36
22,1
97,15


In [9]:
encoded_sequences = [[note_to_idx[pitch] for pitch in sequence] for sequence in all_sequences]
assert len(encoded_sequences) == len(all_sequences)

In [11]:
sequence_length = 64

x_data = []
y_data = []
for sequence in encoded_sequences:
    for i in range(len(sequence) - sequence_length):
        x_data.append(sequence[i:i+sequence_length])
        y_data.append(sequence[i+1:i+1+sequence_length])

assert len(x_data) == len(y_data)